# 17-620 Lab 2 — Encoding Classical Data and Quantum Similarity

**80 minutes | PennyLane | Fall 2026**

### Learning objectives
1. Explain the difference between a classical vector and a quantum feature map.
2. Construct angle and amplitude embeddings with PennyLane.
3. Implement an inverse-circuit overlap kernel and visualize its matrix.
4. Estimate the kernel with finite-shot measurements and discuss practical limitations.

**Textbook:** *A Practical Guide to Quantum Machine Learning and Quantum Optimization*, Chapter 9. **No Qiskit.**

Run the cells in order. The notebook starts with a **fully worked example**. Later sections contain TODOs and checks. Use a new Markdown cell under each conceptual question to record your answer.

> **Key idea:** An encoding is a map `x → |φ(x)⟩`. Different maps produce different similarity measures; a small quantum circuit alone does not establish quantum advantage.

In [ ]:
import pennylane as qml
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

print("PennyLane", qml.__version__)

## Part 1 — Classical baseline (10 minutes)

Our four classical input vectors are rows of `X`. Cosine similarity measures the *direction* between two vectors: +1 is parallel, 0 is orthogonal, −1 is opposite.

The heatmap has **data-point indices** along both axes. Compare the entries involving vectors 0 and 3.

In [ ]:
X = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [-1.0, 0.0]])
cosine = cosine_similarity(X)
print("Classical cosine similarity:\n", np.round(cosine, 3))

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cosine, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(X)), labels=[f"x{i}" for i in range(len(X))])
ax.set_yticks(range(len(X)), labels=[f"x{i}" for i in range(len(X))])
ax.set_title("Classical cosine similarity")
fig.colorbar(im, ax=ax, label="cosine similarity")
plt.show()

**Question 1.** Which pairs are orthogonal? What does a negative cosine similarity mean?

## Part 2 — Angle encoding (20 minutes)

### Worked example: one feature → one qubit

A classical scalar `θ` becomes the angle of an `RY(θ)` rotation applied to `|0⟩`.

```text
classical θ = 0          |0⟩ ──RY(0)─────── measure → 0 with probability 1
classical θ = π/2        |0⟩ ──RY(π/2)───── measure → 0 or 1, equally likely
classical θ = π          |0⟩ ──RY(π)─────── measure → 1 with probability 1
```

The resulting state is `cos(θ/2)|0⟩ + sin(θ/2)|1⟩`. Read the output of this worked cell and compare it with the mathematical prediction.

In [ ]:
worked_dev = qml.device("default.qubit", wires=1)

@qml.qnode(worked_dev)
def worked_rotation(theta):
    qml.RY(theta, wires=0)
    return qml.probs(wires=[0])

for theta in [0.0, np.pi / 2, np.pi]:
    print(f"theta={theta:.3f}: P(0),P(1)={worked_rotation(theta)}")
print(qml.draw(worked_rotation)(np.pi / 2))

### Exercise 2A: two features → two qubits

Now represent each input pair with two Y rotations using `qml.AngleEmbedding`. The function should return the four computational-basis probabilities in the order `00, 01, 10, 11`.

**TODO:** Replace `pass` and then run the checks. If a check fails, inspect your operation and wire order.

In [ ]:
angle_dev = qml.device("default.qubit", wires=2)

@qml.qnode(angle_dev)
def angle_state(x):
    # TODO: replace pass with qml.AngleEmbedding(features=x, wires=[0, 1], rotation="Y")
    pass
    return qml.probs(wires=[0, 1])

angle_inputs = np.array([[0.0, 0.0], [np.pi/2, 0.0],
                         [0.0, np.pi/2], [np.pi/2, np.pi/2]])

print("Example:", angle_state(angle_inputs[0]))
print(qml.draw(angle_state)(angle_inputs[1]))
# Self-check: uncomment after implementing
# assert np.allclose(angle_state(angle_inputs[0]), [1, 0, 0, 0])
# assert np.allclose(angle_state(angle_inputs[3]), [0.25, 0.25, 0.25, 0.25])

**Question 2.** Why are features represented as rotation angles? How would you rescale raw input data to radians? What happens to measurement probabilities if you add `2π` to an angle?

## Part 3 — Quantum overlap kernel (20 minutes)

Rather than measuring each encoded state separately, apply the encoding of `x`, undo the encoding of `y`, and measure the probability of the `|00⟩` outcome.

`|00⟩ ── U(x) ── U(y)† ── measure P(00)`

This probability is **`K(x,y)=|⟨φ(y)|φ(x)⟩|²`**. It is a *squared overlap*, not ordinary signed cosine similarity.

**TODO:** Implement the two encoding steps, compute the kernel matrix, draw a heatmap, and uncomment the checks.

In [ ]:
kernel_dev = qml.device("default.qubit", wires=2)

@qml.qnode(kernel_dev)
def angle_overlap(x, y):
    # TODO: qml.AngleEmbedding for x
    # TODO: qml.adjoint(qml.AngleEmbedding) for y
    return qml.expval(qml.Projector([0, 0], wires=[0, 1]))

def make_kernel(points, circuit):
    return np.array([[circuit(x, y) for y in points] for x in points])

# TODO: uncomment once the function is implemented
# K_angle = make_kernel(angle_inputs, angle_overlap)
# print(np.round(K_angle, 3))
# fig, ax = plt.subplots(figsize=(5, 4))
# im = ax.imshow(K_angle, vmin=0, vmax=1, cmap="viridis")
# ax.set_xticks(range(4), labels=[f"x{i}" for i in range(4)])
# ax.set_yticks(range(4), labels=[f"x{i}" for i in range(4)])
# ax.set_title("Quantum angle-overlap kernel")
# fig.colorbar(im, ax=ax, label="P(00)")
# plt.show()
# assert np.allclose(K_angle, K_angle.T, atol=1e-8)
# assert np.allclose(np.diag(K_angle), np.ones(4), atol=1e-8)
# assert np.all((K_angle >= -1e-8) & (K_angle <= 1 + 1e-8))
# assert np.isclose(angle_overlap(angle_inputs[0], angle_inputs[1]), 0.5)

**Question 3.** Is squared overlap the same as Euclidean distance or classical cosine similarity? Which information disappears when we square an inner product?

## Part 4 — Amplitude embedding (15 minutes)

Here four classical features become four *amplitudes* of a **two-qubit state**. The vector must be normalized. We will compare a squared quantum overlap with a squared classical cosine similarity.

**Important:** This uses few qubits, but preparing an arbitrary amplitude state can be expensive. The circuit output does not prove an end-to-end speedup.

**TODO:** Implement the two amplitude embeddings (second one adjoint), then compare the two matrices.

In [ ]:
amp_dev = qml.device("default.qubit", wires=2)

@qml.qnode(amp_dev)
def amplitude_overlap(a, b):
    # TODO: qml.AmplitudeEmbedding(a, wires=[0, 1], normalize=True)
    # TODO: qml.adjoint(qml.AmplitudeEmbedding)(b, wires=[0, 1], normalize=True)
    return qml.expval(qml.Projector([0, 0], wires=[0, 1]))

A = np.array([[1., 0., 0., 0.], [0., 1., 0., 0.],
              [1., 1., 0., 0.], [-1., 0., 0., 0.]])
expected_amp = cosine_similarity(A)**2
print("Squared classical cosine baseline:\n", np.round(expected_amp, 3))
# TODO: K_amp = make_kernel(A, amplitude_overlap)
# TODO: make a side-by-side or separate heatmap for K_amp
# assert np.allclose(K_amp, expected_amp, atol=1e-7)

**Question 4.** Why do a vector and its negative give the same squared overlap? How does state-preparation cost affect the overall algorithm?

## Part 5 — Sampling the overlap (15 minutes)

The overlap circuit returns an exact `P(00)` on an ideal analytic simulator. Real measurements instead produce bitstrings. Estimate `P(00)` by counting `00` outcomes and dividing by the shot count.

Use `qml.set_shots` on the QNode to avoid deprecated device-shot settings in PennyLane 0.45.1. <https://docs.pennylane.ai/en/stable/code/api/pennylane.workflow.set_shots.html>

**TODO:** Implement the same inverse-encoding steps, then compute the empirical frequency for 1000 shots.

In [ ]:
shots = 1000
sample_dev = qml.device("default.qubit", wires=2)

@qml.set_shots(shots=shots)
@qml.qnode(sample_dev)
def sampled_angle_overlap(x, y):
    # TODO: embed x and undo the embedding of y
    return qml.sample(wires=[0, 1])

x, y = angle_inputs[0], angle_inputs[1]
# TODO: samples = sampled_angle_overlap(x, y)
# TODO: estimate = np.mean(np.all(samples == 0, axis=1))
# TODO: compare to angle_overlap(x, y); expect sampling variation
# assert samples.shape == (shots, 2)
# assert abs(estimate - angle_overlap(x, y)) < 0.12

## Exit ticket (submit as Markdown answers)

1. Exactly what does the quantum overlap kernel measure?
2. Why doesn't logarithmic qubit count for amplitude encoding automatically imply a quantum speedup?
3. How does finite-shot sampling affect the estimated kernel? What happens if we increase shots?
4. Give one fair classical baseline for a future learning task.

**Optional extension:** Train a classifier with a precomputed quantum kernel and compare to a classical kernel on the same train/test split. Do not equate accurate classification with quantum advantage.

**Submission:** Save and submit your completed notebook as instructed in the course LMS.